In [1]:
import os
import torch
import math
import time
import inspect
import tokenizers
import sentencepiece
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sentencepiece as spm
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torchmetrics.text import BLEUScore
from tokenizers import Tokenizer
from datasets import load_dataset, load_from_disk

#tensorboard
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

In [2]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

In [3]:
device_type = "cuda" if device.type.startswith("cuda") else "cpu"

In [4]:
#defualt is highest
torch.set_float32_matmul_precision('high')

In [ ]:
# ====================================================
# CFG
# ====================================================
@dataclass
class CFG:
    d_model = 512
    h=8
    d_ff=2048
    dropout=0.1
    tokenizer_src_loc = './sentencepiece_tok/en_hi_shared.model'
    tokenizer_tgt_loc = './sentencepiece_tok/en_hi_shared.model'
    dataset = 'cfilt/iitb-english-hindi'
    prepared_dataset_dir = './tokenized_dataset/'
    prepared_dataset_name = 'dataset.pt' 
    num_rows_dataset_train =  1650000  # data len: 1659083 #select n rows of dataset as dataset is quite big
    num_rows_dataset_valid = 5000
    src_seq_len = 100
    tgt_seq_len = 120
    src_lang = 'en'
    tgt_lang = 'hi'
    batch_size_train = 128
    batch_size_valid = 1
    valid_interval = 2
    num_workers = 8
    init_learning_rate = .0002
    weight_decay=0.1
    betas = (0.9, 0.95)
    grad_clip = 1.0
    grad_accumulation_steps = 2
    seed=42
    n_epochs=10
    train=True
    experiment_name = 'logs_tensorboard_bpe_tokenizer'
    state_dir = 'models'
    best_state_file_name = 'best_state'
    last_state_file_name = 'last_state'
    best_state = None  # 'best_state_0.pth'
    last_state =  None  # 'last_state_4.pth'
    max_token_gen_len = 100 #total tokens to generate for validation
    print_examples = 3 #total examples to print in validation

In [6]:
class InputEmbeddings(nn.Module):
    def __init__(self, embed_dim: int, vocab_size: int):
        "In paper embed_dim is d_model, which is 512"
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
    def forward(self,x):
        return self.embeddings(x)*math.sqrt(self.embed_dim)

In [7]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, embed_dim: int, seq_len: int, dropout: float):
        super().__init__()        
        self.embed_dim = embed_dim
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(0, seq_len).unsqueeze(1)
        #log exp for numerical stability
        div = torch.exp(-2*(torch.arange(0, embed_dim))/embed_dim*math.log(10000))
        pe = pos*div
        pe[:, 0::2] = torch.sin(pe[:,0::2])
        pe[:, 1::2] = torch.cos(pe[:,1::2])
        pe = pe.unsqueeze(0)
        self.register_buffer('pe',pe)
        
    def forward(self,x):
        
        x = x+(self.pe[:, torch.arange(0, x.shape[1]),:]).requires_grad_(False)
        return self.dropout(x)

In [8]:
class FFN(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
    def forward(self,x):
        return self.feed_forward(x)

In [9]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.h = h
        
        assert d_model%h ==0, "d_model should be divisible by h"
        
        self.d_k = d_model//h
        self.w_q = nn.Linear(d_model,d_model, bias= False)
        self.w_k = nn.Linear(d_model,d_model, bias= False)        
        self.w_v = nn.Linear(d_model, d_model, bias= False)  
        
        self.w_o = nn.Linear(d_model, d_model, bias= False)  
        self.dropout = nn.Dropout(dropout)
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        scores = query@key.transpose(-2,-1)/math.sqrt(d_k)
        #print(query.shape,key.shape, value.shape, scores.shape, mask.shape)
        if mask is not None:
            scores.masked_fill_(mask==0, -1e9)
        scores = scores.softmax(dim=-1)
        
        ### do we need ?
        if dropout is not None:
            scores = dropout(scores)
            
        return scores@value,scores 
        
    def forward(self, q, k, v, mask):
        # batch, seq len, embed dim
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)
        
        # batch, h, seq, d_k
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1,2)
        
        #x, scores = self.attention(query, key, value, mask, self.dropout)
        
        x = F.scaled_dot_product_attention(query, key, value, attn_mask=mask, dropout_p=self.dropout.p)

        x = x.transpose(1,2).contiguous().view(x.shape[0], -1, self.h*self.d_k)
        
        return self.w_o(x)
        

In [10]:
class ResidualConnection(nn.Module):
    
    def __init__(self, d_model:int, dropout:float):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, sublayer):
        #in paper layer norm is applied after the sum 
        return x + self.dropout(sublayer(self.layer_norm(x)))
        

In [11]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model:int, h:int, d_ff:int, dropout:float):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, h, dropout)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.layer_norm_1 = nn.LayerNorm(d_model)
        self.layer_norm_2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        x_ln = self.layer_norm_1(x)
        attention = self.attention(x_ln, x_ln, x_ln, mask)
        x = x + self.dropout(attention)
        
        ffn_out = self.ffn(self.layer_norm_2(x))
        x = x+ self.dropout(ffn_out)
        
        return x
        

In [12]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model:int, h:int, d_ff:int, dropout:float):
        super().__init__()
        self.attention_1 = MultiHeadAttention(d_model, h, dropout)
        self.attention_2 = MultiHeadAttention(d_model, h, dropout)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.layer_norm_1 = nn.LayerNorm(d_model)
        self.layer_norm_2 = nn.LayerNorm(d_model)
        self.layer_norm_3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_out, src_mask, tgt_mask):
        x_ln = self.layer_norm_1(x)
        attention = self.attention_1(x_ln, x_ln, x_ln, tgt_mask)
        x = self.layer_norm_2(x + self.dropout(attention))
        #key, value from encoder and query from decoder
        attention = self.attention_2(x, encoder_out, encoder_out, src_mask)
        x = x + self.dropout(attention)
        
        ffn_out = self.ffn(self.layer_norm_3(x))
        x = x+ self.dropout(ffn_out)
        
        return x
        

In [13]:
class ProjectionLayer(nn.Module):
    def __init__(self, d_model:int, vocab_size: int):
        super().__init__()
        
        self.proj = nn.Linear(d_model, vocab_size)
        
    def forward(self,x):
        #for numerical stability!!!
        return self.proj(x)


In [14]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size:int, src_seq_len:int, tgt_seq_len:int, d_model:int, h:int, d_ff:int, dropout:float, N=6):
        super().__init__()
        
        self.encoder = nn.ModuleList(
            [
                EncoderBlock(d_model, h, d_ff, dropout)
                for _ in range(N)
            ]
        )

        self.decoder = nn.ModuleList(
            [
                DecoderBlock(d_model, h, d_ff, dropout)
                for _ in range(N)
            ]
        )
        
        self.src_embed = InputEmbeddings(d_model, src_vocab_size)
        self.tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)
        self.src_pos_embed = PositionalEmbeddings(d_model, src_seq_len, dropout)
        self.tgt_pos_embed = PositionalEmbeddings(d_model, tgt_seq_len, dropout)
        self.projection_layer = ProjectionLayer(d_model, tgt_vocab_size)
        
    
    def encode(self,x, src_mask):
        x = self.src_embed(x)
        x = self.src_pos_embed(x)
        for layer in  self.encoder:
            x = layer(x, src_mask)
        return x
        
    def decode(self, x, encoder_out,src_mask, tgt_mask):
        x = self.tgt_embed(x)
        x = self.tgt_pos_embed(x)
        for layer in  self.decoder:
            x = layer(x, encoder_out,src_mask,tgt_mask)
        return x
        
    def project(self,x):
        x = self.projection_layer(x)
        return x
    

In [15]:
def initialize_weight(model: Transformer):
    for name,p in model.named_parameters():
        if p.dim()>1:
            nn.init.xavier_uniform_(p)
        else:
            if 'layer' in name:
                continue
            else:
                nn.init.zeros_(p)

In [16]:
config = CFG()

In [ ]:
class BilingualDataset(Dataset):
    def __init__(self, ds, tokenizer_src, tokenizer_tgt, max_len_src, max_len_tgt, src_lang, tgt_lang, dataset_loc, dataset_name):
        self.ds = ds
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.max_len_src = max_len_src
        self.max_len_tgt = max_len_tgt
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.dataset_loc = dataset_loc
        self.dataset_name = dataset_name
        self.sos_token = torch.tensor([tokenizer_src.bos_id()], dtype = torch.int32)
        self.eos_token = torch.tensor([tokenizer_src.eos_id()], dtype = torch.int32)
        self.pad_token = torch.tensor([tokenizer_src.pad_id()], dtype = torch.int32)
        if not os.path.exists(os.path.join(dataset_loc, dataset_name)):
            os.makedirs(dataset_loc, exist_ok=True)
            self.prepare_dataset()
        else:
            data = torch.load(os.path.join(dataset_loc, dataset_name))
            self.encoder_inps = data['encoder_inps']
            self.decoder_inps = data['decoder_inps']
            self.encoder_masks = data['encoder_masks']
            self.decoder_masks = data['decoder_masks']
            self.labels = data['labels']
            self.src_txts = data['src_txts']
            self.tgt_txts = data['tgt_txts']

        del self.ds
        
        
    def __getitem__(self, idx):
        
        return {
            "encoder_inp": self.encoder_inps[idx],
            "decoder_inp": self.decoder_inps[idx],
            "encoder_mask": self.encoder_masks[idx], 
            "decoder_mask": self.decoder_masks[idx],
            "labels": self.labels[idx],
            "src_txt": self.src_txts[idx],
            "tgt_txt": self.tgt_txts[idx]
        }
    
    def prepare_dataset(self):
        print("Preparing dataset...")
        self.encoder_inps = []
        self.decoder_inps = []
        self.encoder_masks = []
        self.decoder_masks = []
        self.labels = []
        self.src_txts = []
        self.tgt_txts = []

        
        for idx in range(len(self.ds)):
            src_txt = self.ds['translation'][idx][self.src_lang]
            tgt_txt = self.ds['translation'][idx][self.tgt_lang]
            
            src_encoded = torch.tensor(self.tokenizer_src.encode(src_txt))
            if len(src_encoded)>self.max_len_src-2:
                src_encoded = src_encoded[:self.max_len_src-2]
            encoder_tokens = torch.ones(self.max_len_src, dtype=torch.int32)*self.pad_token
            encoder_tokens[0]= self.sos_token
            encoder_tokens[1:len(src_encoded)+1] = src_encoded
            encoder_tokens[len(src_encoded)+1] = self.eos_token
            
            tgt_encoded = torch.tensor(self.tokenizer_tgt.encode(tgt_txt))
            if len(tgt_encoded)>self.max_len_tgt-2:
                tgt_encoded = tgt_encoded[:self.max_len_tgt-2]
            decoder_tokens = torch.ones(self.max_len_tgt, dtype=torch.int32)*self.pad_token
            decoder_tokens[0]= self.sos_token
            decoder_tokens[1:len(tgt_encoded)+1] = tgt_encoded
            decoder_tokens[len(tgt_encoded)+1] = self.eos_token 
            label = torch.ones(self.max_len_tgt, dtype=torch.int64)*self.pad_token
            label[:len(tgt_encoded)] = tgt_encoded
            label[len(tgt_encoded)] = self.eos_token

            encoder_mask = (encoder_tokens!=self.pad_token).unsqueeze(0).unsqueeze(0).bool()
            decoder_mask = (((decoder_tokens!=self.pad_token).unsqueeze(-1)* torch.tril(torch.ones(self.max_len_tgt, self.max_len_tgt))).unsqueeze(0)).bool()

            self.encoder_inps.append(encoder_tokens)
            self.decoder_inps.append(decoder_tokens)
            self.encoder_masks.append(encoder_mask)
            self.decoder_masks.append(decoder_mask)
            self.labels.append(label)
            self.src_txts.append(src_txt)
            self.tgt_txts.append(tgt_txt)

        torch.save({
            "encoder_inps": torch.stack(self.encoder_inps), 
            "decoder_inps": torch.stack(self.decoder_inps),
            "encoder_masks": torch.stack(self.encoder_masks),
            "decoder_masks": torch.stack(self.decoder_masks),
            "labels": torch.stack(self.labels),
            "src_txts": self.src_txts,
            "tgt_txts": self.tgt_txts
        }, os.path.join(self.dataset_loc, self.dataset_name))
    
    def __len__(self):
        return len(self.ds)

In [ ]:
def save_model(model, optimizer, epoch, device, state_file_name, state_dir, bleu_score):
    
    if not os.path.exists(state_dir):
        os.makedirs(state_dir)

    if 'best' in state_file_name:
        state_path = os.path.join(state_dir, state_file_name + '.pth')
    else:   
        state_path = os.path.join(state_dir, state_file_name + '_' + str(epoch)+ '.pth')

    # make sure you transfer the model to cpu.
    if device == 'cpu':
        model.to('cpu')

    # save the state_dict
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': torch.tensor(epoch),
        'bleu_score': torch.tensor(bleu_score)
    }, state_path)
    
    #transfer model to gpu again
    if device == 'cuda':
        model.to('cuda')
    
    return

In [19]:
def load_model(model, state_file_name ,state_dir=CFG.state_dir):
    state_path = os.path.join(state_dir, state_file_name)

    # loading the model and getting model parameters by using load_state_dict
    state = torch.load(state_path)
    model.load_state_dict(state['model_state_dict'])
    
    return model

In [20]:
def train(
    train_config: CFG,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    train_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    loss_fn: torch.nn.modules.loss.CrossEntropyLoss,
    vocab_size: int,
    device: torch.device
    )-> float:

    batch_loss = np.array([])
    norms = np.array([])
    model.train()
    batch_iterator = tqdm(train_loader, desc=f"Processing Epoch {epoch_idx}")
    t0 = time.time()
    for step,batch in enumerate(batch_iterator):
        encoder_inp = batch['encoder_inp'].to(device)
        encoder_mask = batch['encoder_mask'].to(device)
        decoder_inp = batch['decoder_inp'].to(device)
        decoder_mask = batch['decoder_mask'].to(device)
        labels = batch['labels'].to(device)
        
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            enoder_out = model.encode(encoder_inp, encoder_mask)
            decoder_out = model.decode(decoder_inp, enoder_out, encoder_mask, decoder_mask)
            project = model.project(decoder_out)
            loss = loss_fn(project.view(-1, vocab_size),labels.view(-1))

        batch_loss = np.append(batch_loss, [loss.item()])
        loss = loss / train_config.grad_accumulation_steps
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.grad_clip)
        norms = np.append(norms, [norm.item()])

        if (step+1)%train_config.grad_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        batch_iterator.set_postfix({"loss": f"{loss.item()*train_config.grad_accumulation_steps:.4f}"})
    dt = time.time() - t0
    tokens_processed = len(train_loader)*(train_config.src_seq_len + train_config.tgt_seq_len)
    tokens_per_sec = tokens_processed/dt
    epoch_loss = batch_loss.mean()
    epoch_norm = norms.mean()
    print('Epoch: {} Train Loss: {:.3f} Average Norm: {:.3f} Tokens/sec: {:.2f} Time: {:.2f}s'.format(epoch_idx, epoch_loss, epoch_norm, tokens_per_sec, dt))
    return epoch_loss

In [ ]:
def validation(
    config: CFG,
    model: Transformer,
    valid_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    max_token_gen_len: int,
    tokenizer_tgt: sentencepiece.SentencePieceProcessor,
    device: torch.device
    )-> float:
    
    '''
        calculate bleu score for translation and display 
        some outputs
    '''
    scores = []
    examples = 0
    eos_token = tokenizer_tgt.eos_id()
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        #batch_iterator = tqdm(valid_loader, desc=f"Processing Epoch {epoch_idx}")
        for batch in valid_loader:
            decoder_inp = torch.empty(1,1).fill_(tokenizer_tgt.bos_id()).type(torch.int32).to(device)
            encoder_inp = batch['encoder_inp'].to(device)
            encoder_mask = batch['encoder_mask'].to(device)
            labels = batch['labels'].to(device)
            enoder_out = model.encode(encoder_inp, encoder_mask)
            for _ in range(max_token_gen_len):
                decoder_mask = torch.tril(torch.ones(decoder_inp.shape[1],decoder_inp.shape[1])).type(torch.bool).to(device)
                decoder_out = model.decode(decoder_inp, enoder_out, encoder_mask, decoder_mask)
                #only last word
                project = model.project(decoder_out[:,-1,:])
                probs = project.softmax(dim=-1)
                next_idx = probs.argmax(dim=-1).item()
                next_idx_tensor = torch.empty(1,1).fill_(next_idx).type(torch.int32).to(device)
                decoder_inp = torch.cat((decoder_inp, next_idx_tensor), dim=1)
                if next_idx == eos_token:
                    break

            tgt_txt = tokenizer_tgt.decode(labels.squeeze().cpu().tolist())
            pred_txt = tokenizer_tgt.decode(decoder_inp.squeeze().cpu().tolist())

            #the bleu score works above 4 gram, modifying the score function for smaller sentences
            sent_len = decoder_inp.shape[1]-2
            if sent_len<4 and sent_len>0:
                bleu_metric = BLEUScore(n_gram=sent_len)
            else:
                bleu_metric = BLEUScore()
            score = bleu_metric([pred_txt], [[tgt_txt]])
            scores.append(score.item())

            if (np.random.random_sample()<.001) and examples<config.print_examples:
                print(f"Source txt: {batch['src_txt'][0]}")
                print(f'Predicted txt: {pred_txt}')
                print(f'Label: {tgt_txt}\n')
                examples +=1
            #batch_iterator.set_postfix({"bleu": f"{score.item():.4f}"})

    print(f"Validation Time: {time.time() - t0:.2f} seconds")

    return np.array(scores).mean()

In [ ]:
def main(
    config: CFG,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    dataset_train: torch.utils.data.Dataset,
    dataset_valid: torch.utils.data.Dataset,
    scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
    summary_writer: SummaryWriter,
    tokenizer_src: sentencepiece.SentencePieceProcessor,
    tokenizer_tgt: sentencepiece.SentencePieceProcessor,
    device: torch.device
    ):
    
    
    train_loader = DataLoader(dataset_train, batch_size=config.batch_size_train,\
                              num_workers=config.num_workers,shuffle=True)
    valid_loader = DataLoader(dataset_valid, batch_size=config.batch_size_valid)
    loss_fn =  nn.CrossEntropyLoss(ignore_index=tokenizer_tgt.pad_id(), label_smoothing=0.1)
    vocab_size = tokenizer_tgt.vocab_size()
    epoch_start = 0
    model.to(device)

    best_bleu = -torch.tensor(np.inf)
    if config.last_state is not None:
        state_path = os.path.join(config.state_dir, config.last_state)
        state = torch.load(state_path)
        model.load_state_dict(state['model_state_dict'])
        best_bleu = state['bleu_score'].item()
        epoch_start = state['epoch'].item() + 1
        print(f"Loaded model from {config.last_state}")
        print(f"Best Bleu Score: {best_bleu:.4f}, Resuming training from epoch {epoch_start}")

    t_begin = time.time()
    for epoch in range(epoch_start,config.n_epochs):
            
        train_loss= train(config, model, optimizer, train_loader, epoch, loss_fn, vocab_size, device)
        summary_writer.add_scalar('Loss/Train',train_loss, epoch) 
        summary_writer.add_scalar('Learning Rate',optimizer.param_groups[0]['lr'] , epoch)

        if ((epoch)%config.valid_interval==0) or epoch == config.n_epochs-1:
        
            bleu_score = validation(config, model, valid_loader, epoch, config.max_token_gen_len,tokenizer_tgt, device)
            print(f'Epoch: {epoch} Bleu Score: {bleu_score:.4f}')
            summary_writer.add_scalar('BleuScore/Valid',bleu_score, epoch)
            if bleu_score > best_bleu:
                best_bleu = bleu_score
                print('...Model Improved. Saving the Model...\n')
                save_model(model, optimizer, epoch,device=device,state_file_name= config.best_state_file_name,
                        state_dir= config.state_dir, bleu_score=bleu_score)
                
            if epoch==config.n_epochs-1:
                save_model(model, optimizer, epoch,device=device,state_file_name= config.last_state_file_name,
                        state_dir= config.state_dir, bleu_score=bleu_score)
                
                
            if scheduler is not None:
                if isinstance(scheduler, ReduceLROnPlateau):
                    scheduler.step(bleu_score)
                else:
                    scheduler.step()
        
        torch.cuda.synchronize()
        elapsed_time = time.time() - t_begin
        speed_epoch = elapsed_time / (epoch + 1)
        eta = speed_epoch *config.n_epochs - elapsed_time

        summary_writer.add_scalar('Time Elapsed',elapsed_time, epoch)
        summary_writer.add_scalar('ETA',eta, epoch)
            
        
    print("Total time: {:.2f}, Best Score: {:.3f}".format(time.time() - t_begin, best_bleu))

In [23]:
config = CFG()

In [24]:
tokenizer_src = spm.SentencePieceProcessor()
tokenizer_src.load(config.tokenizer_src_loc)

True

In [25]:
model = Transformer(
    src_vocab_size= tokenizer_src.vocab_size(),
    tgt_vocab_size= tokenizer_src.vocab_size(),
    src_seq_len=config.src_seq_len,
    tgt_seq_len=config.tgt_seq_len,
    d_model=config.d_model,
    h=config.h,
    d_ff=config.d_ff,
    dropout=config.dropout
    )
initialize_weight(model)

In [26]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params/1e6:.2f}M")

Trainable parameters: 93.29M


In [27]:
def get_optimizer_params(model, lr, weight_decay=0.0):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'lr': lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'lr': lr, 'weight_decay': 0.0}
    ]
    return optimizer_parameters


In [28]:
optimizer_params = get_optimizer_params(model, config.init_learning_rate, config.weight_decay)
fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
use_fused = fused_available and device_type == "cuda" 

optimizer = torch.optim.AdamW(
    optimizer_params,
    lr = config.init_learning_rate,
    betas=config.betas,
    fused=use_fused
)

schedular = ReduceLROnPlateau(optimizer, mode = 'max', factor = 0.3, patience = 2, min_lr= 1e-6)

In [29]:
model.to(device)
model = torch.compile(model)

In [30]:
# if CFG.best_state is not None:
#     model = load_model(model, state_file_name= CFG.best_state, state_dir= CFG.state_dir)

In [ ]:
#ds = load_from_disk(config.data_loc)
ds = load_dataset(config.dataset, split="train")
ds = ds.select(range(config.num_rows_dataset_train + config.num_rows_dataset_valid))
ds = ds.shuffle(seed=config.seed)
ds_train = ds.select(range(config.num_rows_dataset_train))
ds_valid = ds.select(range(config.num_rows_dataset_train,config.num_rows_dataset_train \
                     + config.num_rows_dataset_valid))

#ds = ds.select(range(1000))
train_dataset = BilingualDataset(ds_train, tokenizer_src, tokenizer_src, max_len_src=\
                                config.src_seq_len, max_len_tgt=config.tgt_seq_len,
                                src_lang=config.src_lang, tgt_lang=config.tgt_lang,
                                dataset_loc= config.prepared_dataset_dir, 
                                dataset_name= 'train_'+config.prepared_dataset_name
                                )   

valid_dataset = BilingualDataset(ds_valid, tokenizer_src, tokenizer_src, max_len_src=\
                                config.src_seq_len, max_len_tgt=config.tgt_seq_len,
                                src_lang=config.src_lang, tgt_lang=config.tgt_lang,
                                dataset_loc= config.prepared_dataset_dir, 
                                dataset_name= 'valid_'+config.prepared_dataset_name
                                )

Preparing dataset...
Preparing dataset...


In [ ]:
del ds, ds_train, ds_valid

In [32]:
summary_writer = SummaryWriter(log_dir=config.experiment_name)

In [33]:
%%time
main(config, model, optimizer, train_dataset, valid_dataset,schedular, summary_writer, tokenizer_src,
     tokenizer_src, device)

Processing Epoch 0:   0%|          | 0/12891 [00:00<?, ?it/s]

Epoch: 0 Train Loss: 6.133 Average Norm: 1.800 Tokens/sec: 1200.01 Time: 2363.34s
Source txt: Is He then Who creates like him who does not create? Do you not then mind?
Predicted txt: तो वह पैदा करता है जो जो पैदा करता है जो पैदा करता है? फिर तुम नहीं? फिर तुम मन नहीं?
Label: फिर क्या जो पैदा करता है वह उस जैसा हो सकता है, जो पैदा नहीं करता? फिर क्या तुम्हें होश नहीं होता?

Source txt: The accounts list
Predicted txt: खाता सूची
Label: खाता सूची

Source txt: “My verses were recited to you, so you used to turn back on your heels. ”
Predicted txt: (ऐ रसूल) तुम तुम हमारी आयतें पढ़ी जाती तो तुम तुम (अपनी))))))) तुम (ख़ुदा)))) मुँह फेर रहे थे
Label: (जब) हमारी आयतें तुम्हारे सामने पढ़ी जाती थीं तो तुम अकड़ते किस्सा कहते बकते हुए उन से उलटे पाँव फिर जाते

Validation Time: 555.79 seconds
Epoch: 0 Bleu Score: 0.0497
...Model Improved. Saving the Model...



Processing Epoch 1:   0%|          | 0/12891 [00:00<?, ?it/s]

Epoch: 1 Train Loss: 4.747 Average Norm: 1.474 Tokens/sec: 1200.65 Time: 2362.06s


Processing Epoch 2:   0%|          | 0/12891 [00:00<?, ?it/s]

...

Epoch: 7 Train Loss: 3.277 Average Norm: 1.406 Tokens/sec: 1197.07 Time: 2369.14s


Processing Epoch 8:   0%|          | 0/12891 [00:00<?, ?it/s]

Epoch: 8 Train Loss: 3.200 Average Norm: 1.432 Tokens/sec: 1199.15 Time: 2365.02s
Source txt: exteriorly
Predicted txt: बाहरी
Label: वाह्य रूप-से

Source txt: hemorrhage
Predicted txt: हेमरेज
Label: ब्लीडिंग

Source txt: So I must share my memories about it with the people.
Predicted txt: तो मुझे इसे लोगों के साथ अपनी स्मृतियों को साझा करना चाहिए।
Label: इसलिए उसके विषय में अपनी कुछ स्मृतियाँ लोगों के साथ अवश्य सांझी करुं।

Validation Time: 466.32 seconds
Epoch: 8 Bleu Score: 0.1650
...Model Improved. Saving the Model...



Processing Epoch 9:   0%|          | 0/12891 [00:00<?, ?it/s]

Epoch: 9 Train Loss: 3.142 Average Norm: 1.463 Tokens/sec: 1201.60 Time: 2360.20s
Source txt: Filesystem is busy
Predicted txt: फ़ाइलतंत्र व्यस्त है
Label: फ़ाइलतंत्र व्यस्त है

Source txt: Window Shade Up
Predicted txt: विंडो छाया ऊपर
Label: विंडो शेड अपComment

Source txt: The original construction schedule laid down in 1966 envisaged the completion of the first stage of Bokaro - LRB - 1. 7 million tonnes - RRB - by the end of 1970.
Predicted txt: सन् 1966 में निर्धारित मूल निर्माण अनुसूची में बोकारो के प्रथम चरण (1.7 मिलियन टन) के समापन की परिकल्पना की गई थी.
Label: सन् 1966 में बनी प्रारंभिक निर्माण योजना में बोकारो के प्रथम चरण (17 लाख टन) का निर्माण सन् 1970 के अन्त तक पूरा होने का प्रावधान था.

Validation Time: 458.48 seconds
Epoch: 9 Bleu Score: 0.1665
...Model Improved. Saving the Model...

Total time: 26538.10, Best Score: 0.166
CPU times: user 14h 36min 11s, sys: 6min 3s, total: 14h 42min 15s
Wall time: 7h 22min 18s
